# Baqaee & Farhi (2022) — Calibration Grid (Phase 5)

This notebook runs the full calibration grid: the four nested loops of
`Master_file_3.m` (loop × elasticity × shock_type × htm_share × t_grid).

**Memory note:** Each solve is a 668-variable FD Jacobian system. The full
grid (~230 solves for loop=1, ~96 for loop=2) fits on the host Mac (32 GB)
but may OOM a 5 GB container. Run this notebook on the Mac.

**Output:** CSVs are written to `data/results/` — one folder per cell
(loop/el/st/s/timeseries.csv + sector_prices.csv), plus summary files
(`summary_loop1.csv`, `summary_loop2_htm.csv`, `baseline_fit.csv`).
These CSVs can be loaded in the container for analysis.


In [2]:
# Setup
cd(dirname(Base.active_project()))
using Printf, LinearAlgebra, Statistics
include("src/io_table.jl")
include("src/shocks.jl")
include("src/network.jl")
include("src/model.jl")
include("src/calibration_grid.jl")

println("Modules loaded ✅")


Modules loaded ✅


---
## 1. Single-Cell Test

Run one cell (loop=1, elasticity=1, shock_type=1, htm_share=0) to verify
the driver works before scaling up.


In [3]:
io = load_io_table(joinpath("data", "IO_data_2018.mat"); N=66, year=2015)
shocks = load_shocks("data"; N=66)
sf = build_standard_form(io)

t_grid = [0.0, 0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 1.0]
res, _ = solve_cell(io, shocks, sf; elasticity=1, shock_type=1,
                    htm_share=0.0, t_grid=t_grid)
dR = delta_r_gdp(res)
@printf("Single cell: RGDP=%.4f  \u0394RGDP=%.2f%%  nom=%.4f  retcodes=%s\n", res["GDP"][end], -100*dR[end], res["nominal_GDP"][end], res["retcodes"])


Loading IO table from data/IO_data_2018.mat ... done. N=66, year=2015, beta sum=1.000000
Loading shocks from data ...
  BLS shock: min=-0.5409, max=0.0206
  PCE shock: min=-0.9182, max=0.2089
Building standard form Ω_re: N=66, D=334
  Psi_re[1,2:N+1] sum = 1.807659
Single cell: RGDP=0.9172  ΔRGDP=-8.14%  nom=0.9050  retcodes=[0, 0, 0, 0, 0, 0, 0, 0]


---
## 2. Full Grid (Loop 1)

Run the full loop=1 grid: elasticity=1:2 × shock_type=1:5 × s=1.
This reproduces Figures 2–3 and Tables A1–A2.

> ⚠️ This takes ~5–15 minutes on a 32 GB Mac. Each cell writes its own
> CSV immediately, so partial results survive if interrupted.


In [4]:
outdir = "data/results"
summary = run_calibration_grid(outdir; loops=[1], verbose=true)

println("\n=== Summary (loop=1) ===\n")
println("shock_type | RGDP_bench | Infl_bench | Unemp_bench | RGDP_CD | Infl_CD | Unemp_CD")
for st in 1:5
    @printf("  %-12s | %8.1f%% | %8.1f%% | %9.1f%% | %7.1f%% | %7.1f%% | %8.1f%%\n",
            ["baseline", "supply", "demand", "agg_demand", "supply+sec"][st],
            -summary["RGDP_graph"][1, st],
            summary["Inflation_graph"][1, st],
            summary["Unemp_graph"][1, st],
            -summary["RGDP_graph"][2, st],
            summary["Inflation_graph"][2, st],
            summary["Unemp_graph"][2, st])
end
println(")\nCSVs written to ", joinpath(pwd(), outdir))


Loading IO table from data/IO_data_2018.mat ... done. N=66, year=2015, beta sum=1.000000
Loading shocks from data ...
  BLS shock: min=-0.5409, max=0.0206
  PCE shock: min=-0.9182, max=0.2089
Building standard form Ω_re: N=66, D=334
  Psi_re[1,2:N+1] sum = 1.807659
loop=1 el=1 st=1 s=1 (htm=0.0) | RGDP Δ=-8.13% | n=23 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=1 el=1 st=2 s=1 (htm=0.0) | RGDP Δ=-5.76% | n=23 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=1 el=1 st=3 s=1 (htm=0.0) | RGDP Δ=-5.08% | n=23 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=1 el=1 st=4 s=1 (htm=0.0) | RGDP Δ=-4.28% | n=23 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=1 el=1 st=5 s=1 (htm=0.0) | RGDP Δ=-6.83% | n=23 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=1 el=2 st=1 s=1 (htm=0.0) | RGDP Δ

---
## 3. HtM Sweep (Loop 2)

Run loop=2 (elasticity=1:2, shock_type=1, htm_share=0:0.2:1). This
reproduces Figure 4. Also writes summary CSVs.


In [5]:
htm_summary = run_calibration_grid(outdir; loops=[2], verbose=true)

println("\n=== HtM sweep (loop=2) ===\n")
println("htm_share | RGDP_bench | Infl_bench | Unemp_bench | RGDP_CD | Infl_CD | Unemp_CD")
for (i, sh) in enumerate(0.0:0.2:1.0)
    @printf("  %8.1f | %8.2f%% | %8.2f%% | %9.2f%% | %7.2f%% | %7.2f%% | %8.2f%%\n",
            sh,
            -100*htm_summary["RGDP_graph_htm"][1, i],
            100*htm_summary["Inflation_graph_htm"][1, i],
            100*htm_summary["Unemp_graph_htm"][1, i],
            -100*htm_summary["RGDP_graph_htm"][2, i],
            100*htm_summary["Inflation_graph_htm"][2, i],
            100*htm_summary["Unemp_graph_htm"][2, i])
end


Loading IO table from data/IO_data_2018.mat ... done. N=66, year=2015, beta sum=1.000000


Loading shocks from data ...
  BLS shock: min=-0.5409, max=0.0206
  PCE shock: min=-0.9182, max=0.2089
Building standard form Ω_re: N=66, D=334
  Psi_re[1,2:N+1] sum = 1.807659
loop=2 el=1 st=1 s=1 (htm=0.0) | RGDP Δ=-8.14% | n=9 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=1 st=1 s=2 (htm=0.2) | RGDP Δ=-8.45% | n=9 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=1 st=1 s=3 (htm=0.4) | RGDP Δ=-8.81% | n=9 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=1 st=1 s=4 (htm=0.6) | RGDP Δ=-9.24% | n=9 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0]


┌ Warning: NaN detected in GDP at t=0.995, using previous value
└ @ Main ~/Git/BFRep/(3)BeyondHulten/bf_replication2/src/calibration_grid.jl:259
┌ Warning: NaN detected in nominal GDP at t=0.995, using previous value
└ @ Main ~/Git/BFRep/(3)BeyondHulten/bf_replication2/src/calibration_grid.jl:263
┌ Warning: NaN detected in inflation at t=0.995, using previous value
└ @ Main ~/Git/BFRep/(3)BeyondHulten/bf_replication2/src/calibration_grid.jl:267
┌ Warning: Continuation refinement failed at t=1.0 (htm_share=0.19999999999999996: During the resolution of the non-linear system, the evaluation of the following equation(s) resulted in a non-finite number: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88,

loop=2 el=1 st=1 s=5 (htm=0.8) | RGDP Δ=NaN% | n=9 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 1, 1]
loop=2 el=1 st=1 s=6 (htm=1.0) | RGDP Δ=-10.59% | n=9 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0]


┌ Warning: Second attempt also failed at t=1.0: During the resolution of the non-linear system, the evaluation of the following equation(s) resulted in a non-finite number: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187,

loop=2 el=2 st=1 s=1 (htm=0.0) | RGDP Δ=-8.15% | n=12 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=2 st=1 s=2 (htm=0.2) | RGDP Δ=NaN% | n=12 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]


┌ Warning: Trunc_A dimension mismatch (expected 66×12, got (66, 9)), reinitializing
└ @ Main ~/Git/BFRep/(3)BeyondHulten/bf_replication2/src/calibration_grid.jl:139


loop=2 el=2 st=1 s=3 (htm=0.4) | RGDP Δ=-9.18% | n=12 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=2 st=1 s=4 (htm=0.6) | RGDP Δ=-9.99% | n=12 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=2 st=1 s=5 (htm=0.8) | RGDP Δ=-11.18% | n=12 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
loop=2 el=2 st=1 s=6 (htm=1.0) | RGDP Δ=-12.87% | n=12 pts | retcodes=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

=== HtM sweep (loop=2) ===

htm_share | RGDP_bench | Infl_bench | Unemp_bench | RGDP_CD | Infl_CD | Unemp_CD
       0.0 |     8.14% |    -1.48% |      6.40% |    8.15% |   -1.47% |     6.91%
       0.2 |     8.45% |    -2.70% |      7.11% |     NaN% |     NaN% |      NaN%
       0.4 |     8.81% |    -4.00% |      8.17% |    9.18% |   -3.85% |     9.87%
       0.6 |     9.24% |    -5.42% |      9.00% |    9.99% |   -5.21% |    12.41%
       0.8 |      NaN% |      NaN% |       NaN% |   11.18% |   -6.77% |    15.35%
       1.0 |    10.59% |    -8.71% |     13.43% |   12

---
## 4. Baseline Fit (Parts A, B, D)

The baseline fit CSVs are already written by `run_calibration_grid`.
`baseline_fit.csv` contains sector-level data for the Part-B out-of-sample
fit (PPI, wage, hours), the Part-D tightness/slackness decomposition, and
the Part-A calibration targets.

Use this file in a separate analysis notebook or plotting script.


In [7]:
println("✅ Phase 5 complete. CSVs in ", joinpath(pwd(), outdir))


✅ Phase 5 complete. CSVs in /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication2/data/results
